In [6]:
import pandas as pd
from pathlib import Path

# -------------------------------------------------
# PATHS
# -------------------------------------------------
INPUT_DIR = Path("/Users/jakobwerkgarner/code/mt_dsnow/par_sens/SNOWPACK_data")
OUTPUT_DIR = Path("/Users/jakobwerkgarner/code/mt_dsnow/par_sens/SNOWPACK_data_seasons_daily")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# variables of interest
VARS = ["HS_mod", "HS_meas", "SWE"]


# -------------------------------------------------
# FUNCTIONS
# -------------------------------------------------
def read_smet(file_path):
    """Read SMET file into header and DataFrame."""
    header = []
    fields = None
    data_rows = []
    data_start = False

    with open(file_path, "r") as f:
        for line in f:
            line = line.rstrip()

            if line.startswith("fields"):
                fields = line.split("=", 1)[1].strip().split()

            elif line.strip() == "[DATA]":
                data_start = True
                continue

            elif not data_start:
                header.append(line)

            else:
                if line:
                    data_rows.append(line.split())

    df = pd.DataFrame(data_rows, columns=fields)
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    return header, df


def extract_station_id(header):
    for line in header:
        if line.startswith("station_id"):
            return line.split("=", 1)[1].strip()
    raise ValueError("station_id not found in SMET header")


def snow_season(ts):
    """Snow season Nov–Apr → 'YYYY_YYYY'."""
    if ts.month >= 11:
        return f"{ts.year}_{ts.year + 1}"
    else:
        return f"{ts.year - 1}_{ts.year}"


# -------------------------------------------------
# MAIN LOOP
# -------------------------------------------------
for smet_file in INPUT_DIR.glob("*.smet"):
    print(f"Processing {smet_file.name}")

    header, df = read_smet(smet_file)
    station_id = extract_station_id(header)

    # keep only required variables
    missing = [v for v in VARS if v not in df.columns]
    if missing:
        raise ValueError(f"{smet_file.name} missing variables: {missing}")

    df = df[["timestamp"] + VARS]

    # force numeric, convert nodata to NaN
    df[VARS] = (
        df[VARS]
        .apply(pd.to_numeric, errors="coerce")
        .replace(-999, pd.NA)
    )

    df["station_id"] = station_id
    df["season"] = df["timestamp"].apply(snow_season)

    for season, df_season in df.groupby("season"):
        start_year = int(season.split("_")[0])

        start = pd.Timestamp(start_year, 11, 1)
        end = pd.Timestamp(start_year + 1, 4, 30, 23, 59)

        df_season = df_season[
            (df_season["timestamp"] >= start) &
            (df_season["timestamp"] <= end)
        ]

        if df_season.empty:
            continue

        # -------------------------------------------------
        # DAILY MEAN
        # -------------------------------------------------
        df_daily = (
            df_season
            .set_index("timestamp")[VARS]
            .resample("D")
            .mean()
            .reset_index()
        )

        df_daily["station_id"] = station_id
        df_daily["season"] = season

        out_file = OUTPUT_DIR / f"{station_id}_{season}_daily_HS_SWE.csv"
        df_daily.to_csv(out_file, index=False)

        print(f"  -> written {out_file.name}")

Processing smet_127_2023-09-30T22_00_00.000Z_2025-09-29T22_00_00.000Z_1761646059910.smet
  -> written PCOS2_2023_2024_daily_HS_SWE.csv
  -> written PCOS2_2024_2025_daily_HS_SWE.csv
Processing smet_79_2023-09-30T22_00_00.000Z_2025-09-29T22_00_00.000Z_1761646130483.smet
  -> written LADU2_2023_2024_daily_HS_SWE.csv
  -> written LADU2_2024_2025_daily_HS_SWE.csv
Processing smet_45_2023-09-30T22_00_00.000Z_2025-09-29T22_00_00.000Z_1761645404268_Lizum.smet
  -> written AXLIZ1_2023_2024_daily_HS_SWE.csv
  -> written AXLIZ1_2024_2025_daily_HS_SWE.csv
Processing smet_86_2023-09-30T22_00_00.000Z_2025-09-29T22_00_00.000Z_1761645877080_Rotwandwiesen.smet
  -> written SROT2_2023_2024_daily_HS_SWE.csv
  -> written SROT2_2024_2025_daily_HS_SWE.csv
Processing smet_80_2023-09-30T22_00_00.000Z_2025-09-29T22_00_00.000Z_1761645714097_Timmelsalm.smet
  -> written MTIM2_2023_2024_daily_HS_SWE.csv
  -> written MTIM2_2024_2025_daily_HS_SWE.csv
Processing smet_57_2023-09-30T22_00_00.000Z_2025-09-29T22_00_00.00